In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import glob

# Pasta contendo as anotações e imagens
folder_path = 'data-coral-expert-v1/'

# Criar pasta para salvar as imagens com anotações
output_folder = 'data-coral-expert-v1-plots/'
os.makedirs(output_folder, exist_ok=True)

# Contar imagens se anotação
not_anot = []

# Iterar sobre os arquivos na pasta
for filename in glob.glob("data-coral-expert-v1/**"):
    if filename.endswith('.json'):
        
        try:
            # Caminho do arquivo JSON
            json_path = filename
            with open(json_path) as f:
                data = json.load(f)

            # Determinar o caminho da imagem com base no nome do arquivo JSON
            image_name = os.path.splitext(filename)[0]
            image_extensions = ['.jpg', '.png','.JPG']
            image_path = None
            for ext in image_extensions:
                temp_image_path = os.path.join(image_name + ext)
                if os.path.exists(temp_image_path):
                    image_path = temp_image_path
                    break

            # Se nenhum arquivo de imagem for encontrado, passar para o próximo arquivo JSON
            if image_path is None:
                print(f"Imagem correspondente não encontrada para o arquivo JSON: {filename}")
                continue

            # Carregar a imagem
            img = cv2.imread(image_path)

            # Iterar sobre as anotações no arquivo JSON
            i = 0
            for annotation in data['shapes']:

                '''if annotation["name"] != "Duto_polygon":
                    continue'''

                # Verificar se a anotação é do tipo complex_polygon
                if 'complex_polygon' in annotation:
                    # Carregar os pontos do polígono complexo da anotação atual
                    complex_polygon = annotation['complex_polygon']['path']

                    # Criar uma imagem em branco para desenhar o polígono
                    mask = np.zeros_like(img)

                    # Plotar o polígono complexo
                    for polygon_points in complex_polygon:
                        pts = np.array([[point['x'], point['y']] for point in polygon_points], np.int32)
                        pts = pts.reshape((-1, 1, 2))
                        cv2.polylines(mask, [pts], isClosed=True, color=(255, 0, 0), thickness=1)

                    # Adicionar a máscara à imagem original
                    img = cv2.addWeighted(img, 0.5, mask, 0.5, 0)
                    i+=1
                else:
                    # Carregar os pontos do polígono
                    polygon_points = annotation.get('points', [])
                    
                    # Carregar a bounding box
                    '''bbox = annotation.get('polygon', {}).get('bounding_box', {})
                    x, y, w, h = bbox.get('x', 0), bbox.get('y', 0), bbox.get('w', 0), bbox.get('h', 0)

                    # Desenhar a bounding box na imagem
                    cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), (0, 0, 255), 1)'''

                    # Converter os pontos do polígono para o formato adequado para o OpenCV
                    points = np.array([[int(pt[0]), int(pt[1])] for pt in polygon_points], np.int32)
                    points = points.reshape((-1, 1, 2))
                    
                    # Desenhar o polígono na imagem
                    cv2.polylines(img, [points], isClosed=True, color=(255, 0, 0), thickness=1)
                    i+=1

            if i == 0:
                not_anot.append(filename) 
            # Salvar a imagem com as anotações
            output_path = os.path.join(output_folder, image_name + '_annotated.jpg').replace("data-coral-expert-v1/","")
            cv2.imwrite(output_path, img)
            print(output_path)
        except Exception as exp:
            print(exp)
        
print(len(not_anot))
print(not_anot)

In [ ]:
import os

base_path = "../Coral_Dataset/Segmentação/Test/Anotações/"
for filename in not_anot:
    os.remove(base_path + filename)
    try:
        os.remove(base_path + filename.replace(".json", ".png").replace("Anotações", "Imagens"))
    except:
        os.remove(base_path + filename.replace(".json", ".jpg").replace("Anotações", "Imagens"))


In [ ]:
import glob

for filename_json in glob.glob("../Coral_Dataset/Segmentação/Test/Imagens/**"):
    filename = filename_json.split("/")[-1].split(".")[0]
    achou = False
    for filename_image in glob.glob("../Coral_Dataset/Segmentação/Test/Anotações/**"):
        if filename_image.split("/")[-1].split(".")[0] == filename:
            achou = True
            break
    if not achou:
        try:
            os.remove("../Coral_Dataset/Segmentação/Test/Imagens/" + filename + ".png")
        except:
            os.remove("../Coral_Dataset/Segmentação/Test/Imagens/" + filename + ".jpg")
 
            
    

In [ ]:
import glob

i=0
for filename_json in glob.glob("data-coral-expert-v1-plots/**"):
    i+=1
print(i)

In [ ]:
import zipfile
import os

caminho_arquivo_zip = 'data-coral-expert.zip'
pasta_destino = 'data-coral-expert'

os.makedirs(pasta_destino, exist_ok=True)

with zipfile.ZipFile(caminho_arquivo_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_destino)

print(f'Arquivo {caminho_arquivo_zip} extraído para {pasta_destino}.')
